# 95장 · 검색 모듈의 권한 필터와 빈 결과

검색 모듈에는 질의와 접근할 수 있는 문서 범위를 전달합니다. 검색어가 비어 있는 입력 오류와 검색했지만 결과가 없는 경우를 구분합니다.

공개 사용자의 검색 요청을 예로 들어 장비 필터·본문 검색·출처 반환을 독립 함수로 검증합니다. 빈 검색어는 입력 단계에서 처리합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## retrieve()의 문서 접근 조건

입력 계약　검색어와 신뢰한 테스트 그룹을 받습니다. 지원하지 않는 그룹과 빈 검색어는 오류로 돌려주고, 허용된 문서에서 찾지 못하면 빈 목록을 반환합니다. 노트북에는 가상 데이터가 들어 있으므로 직접 실행해도 외부 서비스로 전송하지 않습니다.


### 셀 01 · retrieve()의 장비 필터

제공 모듈의 부분 문자열 검색과 출처 반환을 작은 입력으로 실행합니다.

제공 기록의 7/7은 검색 모듈을 포함한 전체 응답 함수의 점수입니다. 지금 셀에서는 검색 결과에 RULE-A와 2.1절이 있는지 확인합니다. 이 조회 한 번으로 일곱 답의 정확성을 측정한 것은 아닙니다.


In [1]:
documents = [{'id': 'RULE-A',
  'equipment': '장비 A',
  'text': '점검 전 전원 표시등을 확인합니다.',
  'source': '가상 장비실 규정 A',
  'page': 1,
  'section': '2.1',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['public', 'admin']},
 {'id': 'RULE-B',
  'equipment': '장비 B',
  'text': '점검 전 고정 장치를 확인합니다.',
  'source': '가상 장비실 규정 B',
  'page': 2,
  'section': '2.2',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['public', 'admin']},
 {'id': 'RULE-ADMIN',
  'equipment': None,
  'text': '관리자 전용 절차는 선임 검토 기록을 확인하는 것입니다.',
  'source': '가상 장비실 내부 규정',
  'page': 3,
  'section': '3.1',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['admin']}]
def retrieve(query, group):
    if group not in ("public", "admin"):
        raise PermissionError("확인되지 않은 그룹입니다.")
    if not isinstance(query, str) or not query.strip():
        raise ValueError("검색어를 입력하세요.")
    query = query.strip()
    allowed = [doc for doc in documents if group in doc["groups"]]
    fields = ("id", "text", "source", "page", "section", "revised_on", "version")
    return [{k: doc[k] for k in fields} for doc in allowed
            if query in doc["text"] or query == doc["equipment"]]

hit = retrieve("장비 A", "public")[0]
print(hit["text"], hit["id"], hit["section"])

점검 전 전원 표시등을 확인합니다. RULE-A 2.1


점검 문장과 RULE-A, 2.1이 이어집니다. 91장과 달리 반환한 값만으로 원문 위치를 표시할 수 있습니다. 팀이 벡터 검색을 택하더라도 권한에 맞는 후보와 반환 필드는 그대로 지킬 수 있습니다.

RULE-A를 반환한 근거에는 장비와 규정 위치가 함께 있습니다. 검색어가 비었을 때 임의의 첫 규정을 돌려주면 정상 결과처럼 보이면서 대상 장비를 잘못 안내할 수 있습니다. 빈 검색어는 별도 입력 오류로 처리합니다.


#### retrieve()에서 공개 범위를 적용한 뒤 검색하는 순서

```mermaid
flowchart TD
ngroup["그룹 검사"]
  nquery["검색어 검사"]
  nallowed["공개 범위"]
  nmatch["본문·장비 대조"]
  nhit["결과 필드"]
  ngroup --> nquery
  nquery --> nallowed
  nallowed --> nmatch
  nmatch --> nhit
```

장비 A를 검색하는 동안 처음 세 문서가 허용 문서 두 건을 거쳐 결과 한 건으로 줄어듭니다.

## 빈 입력 오류와 미적중 결과

모듈은 근거를 찾으면 문서 목록을 반환합니다. 95장 전체 응답 함수에서는 교체·일정 문의를 받으면 no_evidence로 마치고 담당자 확인을 안내합니다. 검색 후보가 없을 때도 같은 안내를 쓸 수 있지만 함수 입력이 잘못됐을 때는 호출 코드를 고쳐야 합니다.


### 셀 02 · 빈 검색어 예외

사용자가 누른 공백과 자료가 없는 상태를 구분해 보세요.


In [2]:
try:
    retrieve(" ", "public")
except ValueError as error:
    print(error)

검색어를 입력하세요.


“검색어를 입력하세요.”가 나옵니다. 호출 전 입력을 고치면 재시도할 수 있고, 문서 내용을 바꿀 필요가 없습니다.


#### 빈 검색어와 검색 실패 구별하기

<mark>유효한 질문을 받았지만 접근 가능한 문서에 근거가 없는 경우를 따로 다룹니다.</mark> 허용 그룹만 바꿔 같은 장비를 질의해 보세요. 권한 밖 문서의 ID나 문장이 빈 결과 안내에 섞이는지도 확인합니다.

선수 개념 · 89장 자료 접근 범위와 대체 데이터

### 과제 03 · 선택 연습 · 허용 문서 범위

공개 그룹으로 관리자 절차를 검색한 결과를 확인하고 사용자에게 보여 줄 다음 문장을 써 보세요.


In [3]:
# retrieve에 검색어와 public을 전달하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>matches = retrieve(&quot;관리자 전용 절차&quot;, &quot;public&quot;)
print(matches)
if not matches:
    print(&quot;주어진 문서에서 확인할 수 없습니다. 담당자에게 확인하세요.&quot;)</code></pre>
<p>실행 결과</p><pre><code>[]
주어진 문서에서 확인할 수 없습니다. 담당자에게 확인하세요.</code></pre>
<p>빈 목록이면 내부 문서의 이름이나 절을 노출하지 않고 다음 행동을 안내할 수 있습니다. 권한은 실제 서비스에서 서버가 정합니다.</p>
</details>


## 조회 함수를 공개용 도구로 감싸고 노출된 인자 확인하기

독립적으로 동작하던 retrieve를 팀의 메시지 처리기에서 호출하려고 합니다. 공개 시연 경로의 도구가 질문마다 접근 그룹을 다시 입력받으면 그룹을 결정한 주체가 불분명해집니다.

### 셀 05 · 공개 경로에서 사용할 검색 도구

이 실습에서 public은 서버 측 공개 시연 경로의 고정 그룹입니다. LangChain 도구에는 검색어 한 개만 노출하고 원문 레코드는 artifact에 보관합니다.

In [4]:
from langchain_core.tools import tool

@tool(response_format="content_and_artifact")
def public_rules(query: str):
    """공개 장비 규정에서 검색어와 관련된 절을 찾습니다."""
    matches = retrieve(query, "public")
    return "\n".join(item["text"] for item in matches), matches

tool_request = {"type": "tool_call", "id": "cap-95", "name": public_rules.name,
                "args": {"query": "장비 B"}}
answer_message = public_rules.invoke(tool_request)
print(list(public_rules.args), answer_message.tool_call_id, answer_message.artifact[0]["id"])

['query'] cap-95 RULE-B


query만 인자 목록에 있고 호출 cap-95가 RULE-B를 찾았습니다. 이 예시에서 그룹은 공개 도구 함수 안에서 정해집니다. 서로 다른 사용자를 실제 인증하는 기능은 별도의 서버 경계가 맡습니다.

### 도구의 반환 계약을 팀 자료로 검산하기

같은 호출에서 query를 “교체 예정일”로 바꾸세요. 결과 레코드가 비면 첫 항목을 꺼내지 말고 현재 호출 ID와 빈 목록을 함께 기록합니다. 앞쪽의 빈 문자열 오류가 발생한 경우와 이 기록을 나란히 보세요.

<mark>도구 명세에 쓰는 인자는 호출자가 선택해도 되는 값만 포함합니다.</mark> 질문을 다시 쓰도록 허용해도 조회 그룹까지 요청 본문의 자유 입력으로 정할 필요는 없습니다.

96장 화면에는 artifact의 첫 문장뿐 아니라 찾아간 규정의 실제 절·쪽·판본을 건넵니다.

## 돌아보기

장비 A 검색, 빈 입력 오류, 공개 그룹에서 후보 없음까지가 이 노트북에서 직접 관찰한 세 결과입니다. 96장에는 그중 한 건의 입력과 반환값을 넘기세요. 제공된 전체 응답 평가(eval-v1)의 별도 기록은 thin 2/7, module 7/7입니다.
